In [ ]:
""" 
Proposed by Jules Nde, PhD, Washburn Lab, Cancer Biology, University of Kansas Medical Center (25/8/2025)

Author: Jules Nde, Washburn Lab, Cancer Biology, University of Kansas Medical Center
Copyright © 2025 by Jules Nde. All rights reserved. No part of this work may be reproduced or transmitted in any form without permission.

This script Assigns different chain IDs to specified residue ranges of a given complex PDB file, then renumber those chains from 1.

    1-) input_directory: Path to the input PDB files directory.
    2-) output_directory Path to the folder to save output PDBs
    3-) The given cahin, default chain A
    4-) residue_ranges: List of tuples, where each tuple contains:
                            - (start_residue, end_residue, chain_id)
    5-) output_file: Path to the output PDB file.
"""

import os
from Bio import PDB

def assign_chains_to_multiple_ranges(pdb_file, residue_ranges, output_file, target_chain_id):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('structure', pdb_file)

    new_model = PDB.Model.Model(0)

    for model in structure:
        chain_residue_map = {}

        for chain in model:
            if chain.id == target_chain_id:
                # Split and assign residues to new chains based on the ranges
                for residue in chain:
                    hetfield, resseq, icode = residue.get_id()

                    if hetfield != ' ':
                        continue  # Skip heteroatoms like water or ions

                    assigned_chain_id = None
                    for (start_residue, end_residue, new_chain_id) in residue_ranges:
                        if start_residue <= resseq <= end_residue:
                            assigned_chain_id = new_chain_id
                            break

                    if assigned_chain_id is None:
                        continue  # Skip residues outside defined ranges

                    if assigned_chain_id not in chain_residue_map:
                        chain_residue_map[assigned_chain_id] = []

                    chain_residue_map[assigned_chain_id].append(residue.copy())
            else:
                # Copy other chains unchanged
                new_chain = PDB.Chain.Chain(chain.id)
                for residue in chain:
                    new_chain.add(residue.copy())
                new_model.add(new_chain)

        for chain_id, residues in chain_residue_map.items():
            new_chain = PDB.Chain.Chain(chain_id)
            for idx, residue in enumerate(residues, start=1):
                hetfield, _, icode = residue.get_id()
                new_residue_id = (hetfield, idx, icode)
                residue.id = new_residue_id
                new_chain.add(residue)
            new_model.add(new_chain)

    new_structure = PDB.Structure.Structure('modified')
    new_structure.add(new_model)

    io = PDB.PDBIO()
    io.set_structure(new_structure)
    io.save(output_file)

def process_pdb_directory(input_dir, output_dir, residue_ranges, target_chain_id):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    pdb_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdb")]

    if not pdb_files:
        print("No PDB files found in the input directory.")
        return

    for pdb_file in pdb_files:
        input_path = os.path.join(input_dir, pdb_file)
        output_path = os.path.join(output_dir, f"{os.path.splitext(pdb_file)[0]}_chains.pdb")
        assign_chains_to_multiple_ranges(input_path, residue_ranges, output_path, target_chain_id)

# === USER CONFIGURATION ===

input_directory = "Rpd3L_Complex_Analysis/core_Cti6_docked_Ume6"    # Folder containing input PDBs
output_directory = "Rpd3L_Complex_Analysis/core_Cti6_docked_Ume6"      # Folder to save output PDBs
target_chain_id = "A"             # The chain to split

# Define residue ranges and the new chain IDs

residue_ranges = [
    (1, 940, 'A'), (941, 1876, 'S'), (1877, 2309, 'C'), (2310, 2742, 'D'), (2743, 3172, 'E'),
    (3173, 3502, 'F'), (3503, 3907, 'G'), (3908, 4234, 'H'), (4235, 4435, 'I'), (4436, 4729, 'J'),
    (4730, 5325, 'K'), (5326, 5785, 'M'), (5786, 6385, 'L'), (6386, 6845, 'N'),(6846, 7351, 'O')
]

# Run the batch processor
process_pdb_directory(input_directory, output_directory, residue_ranges, target_chain_id)
